In [1]:
import os, json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv()

True

In [2]:
with open("bajaj_db.json") as f:
    db = json.load(f)
    

In [3]:
# step 1: define your tool( your python functions)


@tool
def get_loan_status(loan_id: str) -> dict:
    """Fetches current status of a Bajaj Finance loan from the database.
    Returns EMI amount, remaining months, outstanding balance, next due date.
    Use this when customer asks about their loan details, EMI, or balance.
    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found"}
    loan = db["loans"][loan_id]
    return {
        "customer_name": loan["customer_name"],
        "emi": loan["emi"],
        "remaining_months": loan["remaining_months"],
        "outstanding": loan["outstanding"],
        "next_due_date": loan["next_due_date"],
        "prepayment_charge_pct": loan["prepayment_charge_pct"]
    }



In [4]:
# step 2 : bind your list of tools to your llm
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [get_loan_status]
llm_with_tools = llm.bind_tools(tools)


In [5]:
llm_with_tools.invoke('hello')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 104, 'total_tokens': 114, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9075db19fa', 'id': 'chatcmpl-DbdnrY8oQUb24Eru7p2mwLikDV7zl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019df0ec-4a56-7091-8829-b8dba21c4fde-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 104, 'output_tokens': 10, 'total_tokens': 114, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
# Customer message
system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="what is the status of loan id BFL9988")

message = [system,user_msg]
response = llm_with_tools.invoke(message)
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 129, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c7625e91ee', 'id': 'chatcmpl-DbdotWc1ib0beNKx9d81nPEConNs6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df0ed-42f3-7423-b55b-00afd7f8a9c6-0', tool_calls=[{'name': 'get_loan_status', 'args': {'loan_id': 'BFL9988'}, 'id': 'call_Ojx5duXkL9X4CgaUspp8qMh2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 129, 'output_tokens': 20, 'total_tokens': 149, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [10]:
response.tool_calls

[{'name': 'get_loan_status',
  'args': {'loan_id': 'BFL9988'},
  'id': 'call_Ojx5duXkL9X4CgaUspp8qMh2',
  'type': 'tool_call'}]

In [11]:
tool_map={"get_loan_status":get_loan_status
}

tool_result = []

for tc in response.tool_calls:
    tool_name = tc["name"]
    print("tool name: ",tool_name)
    tool_args = tc["args"]
    print("tool_args ",tool_args)
    tool_id = tc["id"]

    # call your tool --> execute ??
    result = tool_map[tool_name].invoke(tool_args)
    # print('result from tools',result)
    tool_result.append((tool_id,tool_name,result))

tool name:  get_loan_status
tool_args  {'loan_id': 'BFL9988'}


In [12]:
tool_result

[('call_Ojx5duXkL9X4CgaUspp8qMh2',
  'get_loan_status',
  {'customer_name': 'Priya Mehta',
   'emi': 24500,
   'remaining_months': 204,
   'outstanding': 2180000,
   'next_due_date': '2026-05-10',
   'prepayment_charge_pct': 0.0})]

In [13]:
# step 4: pass the result to llm with Tool message --> get the final user ou

message = [system,user_msg,response]

for tool_id,tool_name, result in tool_result:
    tool_msg = ToolMessage(
        content = str(result) ,# must be in string
        tool_call_id = tool_id

    )

    message.append(tool_msg)



In [14]:
message

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is the status of loan id BFL9988', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 129, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c7625e91ee', 'id': 'chatcmpl-DbdotWc1ib0beNKx9d81nPEConNs6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df0ed-42f3-7423-b55b-00afd7f8a9c6-0', tool_calls=[{'name': 'get_loan_status', 'args': {'loan_id': 'BFL9988'}, 'id': 'call_Ojx5duXkL9X4CgaUspp

In [15]:
# step 5 : pass entire messages --> llm

final_response = llm_with_tools.invoke(message)

In [16]:
final_response

AIMessage(content='The status of loan ID BFL9988 is as follows:\n\n- **Customer Name:** Priya Mehta\n- **EMI Amount:** ₹24,500\n- **Remaining Months:** 204\n- **Outstanding Balance:** ₹2,180,000\n- **Next Due Date:** May 10, 2026\n- **Prepayment Charge Percentage:** 0.0%\n\nIf you have any further questions or need assistance, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 217, 'total_tokens': 313, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c7625e91ee', 'id': 'chatcmpl-DbdtQxL4JUPEeHqtzYgWzlfu4c553', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019df0f1-8e2f-7711-ab0d-8a4d4d96af86-0', tool_calls=[], invalid_too